In [ ]:
# Notebook 1: Data Loading & Conversation Building

import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU found – go to Runtime > Change runtime type > T4 GPU')

!pip install -q qdrant-client sentence-transformers vaderSentiment textstat xgboost lightgbm imbalanced-learn scikit-learn matplotlib seaborn plotly tqdm

import os, gc, re, json, pickle, warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from pathlib import Path
warnings.filterwarnings('ignore')
tqdm.pandas()

from google.colab import drive
drive.mount('/content/drive')

# Paths
RAW_CSV = '/content/drive/MyDrive/AIE_project/twcs.csv'
OUT_DIR = Path('/content/drive/MyDrive/AIE_project/outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONVERSATIONS_PATH = OUT_DIR / 'conversations.csv'
print('Output directory:', OUT_DIR)

Tue Apr 21 12:31:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   35C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
CHUNK_SIZE = 100_000
COLS = ['tweet_id', 'author_id', 'inbound', 'created_at', 'text',
        'response_tweet_id', 'in_response_to_tweet_id']

chunks = []
total_rows = 0
inbound_rows = 0

print('Loading customer tweets...')
for chunk in tqdm(pd.read_csv(RAW_CSV, usecols=COLS, chunksize=CHUNK_SIZE,
                               dtype={'tweet_id': str, 'author_id': str,
                                      'inbound': bool,
                                      'response_tweet_id': str,
                                      'in_response_to_tweet_id': str})):
    total_rows += len(chunk)
    inbound_chunk = chunk[chunk['inbound'] == True].copy()
    inbound_rows += len(inbound_chunk)
    chunks.append(inbound_chunk)

df_customers = pd.concat(chunks, ignore_index=True)
del chunks
gc.collect()
print(f'Customer rows: {len(df_customers):,}')

Loading customer tweets...


0it [00:00, ?it/s]

Customer rows: 1,537,843


In [ ]:
company_chunks = []
for chunk in tqdm(pd.read_csv(RAW_CSV,
                              usecols=['tweet_id', 'inbound', 'text',
                                       'in_response_to_tweet_id'],
                              chunksize=CHUNK_SIZE,
                              dtype={'tweet_id': str,
                                     'in_response_to_tweet_id': str,
                                     'inbound': bool})):
    company_chunk = chunk[chunk['inbound'] == False][[
        'tweet_id', 'text', 'in_response_to_tweet_id'
    ]].copy()
    company_chunks.append(company_chunk)

df_company = pd.concat(company_chunks, ignore_index=True)
df_company.rename(columns={
    'tweet_id': 'reply_tweet_id',
    'text': 'company_reply',
    'in_response_to_tweet_id': 'tweet_id'
}, inplace=True)
del company_chunks
gc.collect()
print(f'Company reply rows: {len(df_company):,}')

0it [00:00, ?it/s]

Company reply rows: 1,273,931


In [ ]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    text = re.sub(r'http\S+|www\.\S+', '', text)
    text = re.sub(r'@\S+', '', text)
    text = re.sub(r'^RT\s+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print('Cleaning customer text...')
df_customers['text_clean'] = df_customers['text'].progress_apply(clean_text)
df_customers = df_customers[df_customers['text_clean'].str.len() > 5].copy()

# Join with company replies (keep first reply per tweet)
df_company_dedup = df_company.drop_duplicates(subset='tweet_id', keep='first')
df_conv = df_customers.merge(
    df_company_dedup[['tweet_id', 'company_reply']],
    on='tweet_id', how='left'
)

def build_conversation(row):
    customer_part = f"Customer: {row['text_clean']}"
    if pd.notna(row['company_reply']):
        reply_clean = clean_text(str(row['company_reply']))
        return f"{customer_part}\nSupport: {reply_clean}"
    return customer_part

df_conv['conversation'] = df_conv.progress_apply(build_conversation, axis=1)

# Save
df_conv.to_csv(CONVERSATIONS_PATH, index=False)
print(f'Saved conversations to {CONVERSATIONS_PATH}')
print(f'Shape: {df_conv.shape}')

Cleaning customer text...


  0%|          | 0/1537843 [00:00<?, ?it/s]

  0%|          | 0/1503227 [00:00<?, ?it/s]

Saved conversations to /content/drive/MyDrive/AIE_project/outputs/conversations.csv
Shape: (1503227, 10)
